# SQL Analytics

## Objective


Use the dimensional MySQL model developed in the previous stages to answer the project's business questions through analytical SQL queries.

This stage focuses on descriptive and exploratory analysis of well production, hydraulic fracturing, geological, operational, temporal, and spatial data.

The analysis follows the structure:

**Business Question → SQL Query → Result → Interpretation → Business Conclusion**

## Analytical Scope

The SQL analysis will address the business questions defined in the Business Understanding stage.

The analysis is organized into six analytical areas:

1. Production Performance
2. Temporal Analysis
3. Hydraulic Fracturing
4. Geological Analysis
5. Operational Analysis
6. Spatial Analysis

The analysis will use the dimensional model already implemented in MySQL.

No additional data loading, ETL, or modification of the fact tables will be performed in this stage.

## Dimensional Model

The SQL analysis will be performed using the following dimensional model:

### Fact Tables

- `fact_production`
- `fact_fracturing`

### Dimension Tables

- `dim_time`
- `dim_well`
- `dim_geology`
- `dim_field_area`
- `dim_location`
- `dim_company`

The fact tables contain the analytical measurements, while the dimension tables provide the temporal, well, geological, spatial, and company attributes required to analyze production and fracturing performance.

## Analytical Principles

The SQL analysis will follow these principles:

- Use only variables available in the dimensional model.
- Preserve the definitions established during the Business Understanding stage.
- Prefer transparent and reproducible SQL queries.
- Distinguish descriptive patterns from causal relationships.
- Report limitations when data coverage or completeness affects interpretation.
- Avoid unsupported assumptions or artificial imputation.
- Validate analytical results before drawing business conclusions.

The purpose of this stage is to identify patterns and associations that can later be investigated through Python-based statistical analysis.

## MySQL Connection

In [1]:
import pandas as pd
import mysql.connector

In [2]:
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="123456789"
)

cursor = conn.cursor(buffered=True)
cursor.execute("USE oil_gas_dw")

print("Database selected.")

Database selected.


## BQ-01 — How does well production evolve over time?

### Business Question

How does well production evolve over time?

### Analytical Approach

Production will be aggregated monthly to evaluate the temporal evolution of oil and gas production.

The production fact table will be joined with the time dimension through `time_key`.

### SQL Query

In [3]:
query_bq01 = """
SELECT
    t.year,
    t.month,
    SUM(fp.prod_pet) AS total_oil_production,
    SUM(fp.prod_gas) AS total_gas_production
FROM fact_production AS fp
JOIN dim_time AS t
    ON fp.time_key = t.time_key
GROUP BY
    t.year,
    t.month
ORDER BY
    t.year,
    t.month;
"""

### Result

In [4]:
df_bq01 = pd.read_sql(query_bq01, conn)

df_bq01.head()

C:\Users\Admin\AppData\Local\Temp\ipykernel_13132\2014480578.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_bq01 = pd.read_sql(query_bq01, conn)


,year,month,total_oil_production,total_gas_production
0,2006,1,1070.01,32508.00
1,2006,2,994.52,30792.89
2,2006,3,1136.85,33042.57
3,2006,4,1113.70,31362.11
4,2006,5,1505.22,39218.74


Control 1: Month of maximum and minimum production

In [5]:
query_control_01 = """
SELECT
    min(t.full_date),
    max(t.full_date)
FROM dim_time AS t
JOIN fact_production AS fp
    ON fp.time_key = t.time_key
"""

df_control_01 = pd.read_sql(query_control_01, conn)
df_control_01

C:\Users\Admin\AppData\Local\Temp\ipykernel_13132\2134265284.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_control_01 = pd.read_sql(query_control_01, conn)


,min(t.full_date),max(t.full_date)
0,2006-01-31,2026-06-30


Control 2: Months with no production records

In [6]:
query_control_02 = """
SELECT
    cal.year,
    cal.month
FROM (
    SELECT DISTINCT
        year,
        month
    FROM dim_time
    WHERE full_date <= (
        SELECT MAX(t2.full_date)
        FROM dim_time AS t2
        JOIN fact_production AS fp2
            ON t2.time_key = fp2.time_key
    )
) AS cal
LEFT JOIN (
    SELECT DISTINCT
        t.year,
        t.month
    FROM fact_production AS fp
    JOIN dim_time AS t
        ON fp.time_key = t.time_key
) AS prod
    ON cal.year = prod.year
    AND cal.month = prod.month
WHERE prod.year IS NULL
ORDER BY
    cal.year,
    cal.month;
"""

df_control_02 = pd.read_sql(query_control_02, conn)
df_control_02

C:\Users\Admin\AppData\Local\Temp\ipykernel_13132\2095310745.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_control_02 = pd.read_sql(query_control_02, conn)


,year,month


### Interpretation

The monthly production of oil and gas can be analyzed from January 2006 to June 2026.

Control 01 confirmed the actual production period, while Control 02 confirmed that there are no complete months without production records within this period.

However, these results are descriptive and do not allow us to determine why production increases or decreases. The BQ-01 analysis describes the overall monthly production trend across all wells but does not identify the factors associated with changes in production.

### Business Conclusion

Recorded production shows a continuous monthly time series from January 2006 to June 2026, with no complete months missing from the production records within this period.

## BQ-02 Which wells achieve the highest cumulative production?

### Business Question

Which wells achieve the highest cumulative production?

### Analytical Approach

Calculate cumulative oil and gas production for each well by aggregating all available production records across the well's production history.

Oil and gas production will be analyzed separately because they are measured in different units.

The wells will then be ranked by cumulative production to identify those with the highest recorded oil and gas output.

### SQL

In [7]:
query_bq02_oil = """
SELECT 
    well_key,
    SUM(prod_pet) AS cumulative_oil_production,
    SUM(prod_gas) AS cumulative_gas_production
FROM fact_production
GROUP BY well_key
HAVING cumulative_oil_production != 0 
    OR cumulative_gas_production != 0
ORDER BY cumulative_oil_production DESC
LIMIT 10;
"""

### Result

In [8]:
df_bq02_oil = pd.read_sql(query_bq02_oil, conn)
df_bq02_oil

C:\Users\Admin\AppData\Local\Temp\ipykernel_13132\1618429688.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_bq02_oil = pd.read_sql(query_bq02_oil, conn)


,well_key,cumulative_oil_production,cumulative_gas_production
0,8688,337398.08,29323.18
1,7123,243289.05,39119.97
2,9840,234654.13,107702.32
3,6462,229963.66,24752.81
4,6931,219702.87,18918.53
5,8562,213812.54,10623.53
6,8629,198351.26,28169.87
7,8645,194244.57,26474.44
8,7120,193246.17,31979.36
9,7795,184403.40,18092.88


In [9]:
query_bq02_gas = """
SELECT 
    well_key,
    SUM(prod_pet) AS cumulative_oil_production,
    SUM(prod_gas) AS cumulative_gas_production
FROM fact_production
GROUP BY well_key
HAVING cumulative_oil_production != 0 
    OR cumulative_gas_production != 0
ORDER BY cumulative_gas_production DESC
LIMIT 10;
"""

In [10]:
df_bq02_gas = pd.read_sql(query_bq02_gas, conn)
df_bq02_gas

C:\Users\Admin\AppData\Local\Temp\ipykernel_13132\1302759985.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_bq02_gas = pd.read_sql(query_bq02_gas, conn)


,well_key,cumulative_oil_production,cumulative_gas_production
0,9943,705.70,743837.25
1,9948,320.08,680255.15
2,9915,3633.88,659744.37
3,9885,6391.28,651211.48
4,9924,1404.09,638858.00
5,8930,14042.73,633618.58
6,9976,107.93,589759.14
7,9139,8445.96,558050.32
8,9957,121.35,538323.81
9,8899,21682.42,531463.76


### Interpretation

The results show that the wells with the highest cumulative oil production are not necessarily the same wells that achieve the highest cumulative gas production.

The oil ranking is led by well 8688, with 337,398.08 units of cumulative oil production, while the gas ranking is led by well 9943, with 743,837.25 units of cumulative gas production.

The gas ranking also includes several wells with relatively low cumulative oil production. For example, well 9943 accumulated 743,837.25 units of gas but only 705.70 units of oil.

These results indicate that well production performance varies depending on the hydrocarbon considered. However, the rankings are descriptive and do not establish the geological or operational factors associated with these differences.

### Business Conclusion

The analysis shows that the wells with the highest cumulative oil production are not necessarily the same wells with the highest cumulative gas production. This indicates that well performance differs depending on the hydrocarbon considered. The results provide an initial identification of high-performing wells and establish a basis for further analysis of the geological and operational factors associated with these production patterns.

## BQ-03 Which wells exhibit the fastest production decline over time?

### Business Question

Which wells exhibit the fastest production decline over time?

### Analytical Approach

Analyze the production history of each well over time to identify wells that experience the fastest decline in oil and gas production.

For each well, production records will be ordered chronologically and the change between the beginning and the end of its production history will be evaluated.

Oil and gas production will be analyzed separately because they are measured in different units.

The decline will be evaluated relative to the duration of the available production history so that wells with different observation periods can be compared more consistently.

### SQL

In [11]:
query_bq03_oil = """
SELECT
    well_key,
    first_date,
    last_date,
    duration_months,
    first_prod_pet,
    last_prod_pet,
    oil_decline_absolute,
    oil_decline_pct,
    oil_decline_pct_per_month
FROM (
    SELECT
        first_prod.well_key,        
        first_prod.first_date,
        first_prod.first_prod_pet,        
        last_prod.last_date,
        last_prod.last_prod_pet,        
        TIMESTAMPDIFF(
            MONTH,
            first_prod.first_date,
            last_prod.last_date
        ) AS duration_months,        
        first_prod.first_prod_pet - last_prod.last_prod_pet
            AS oil_decline_absolute,        
        CASE
            WHEN first_prod.first_prod_pet > 0
            THEN (
                (first_prod.first_prod_pet - last_prod.last_prod_pet)
                / first_prod.first_prod_pet
            ) * 100
            ELSE NULL
        END AS oil_decline_pct,        
        CASE
            WHEN first_prod.first_prod_pet > 0
                AND TIMESTAMPDIFF(
                    MONTH,
                    first_prod.first_date,
                    last_prod.last_date
                ) > 0
            THEN (
                (
                    first_prod.first_prod_pet - last_prod.last_prod_pet
                )
                / first_prod.first_prod_pet
            ) * 100
            / TIMESTAMPDIFF(
                MONTH,
                first_prod.first_date,
                last_prod.last_date
            )
            ELSE NULL
        END AS oil_decline_pct_per_month
    FROM (
        SELECT
            fp.well_key,
            dt.full_date AS first_date,
            fp.prod_pet AS first_prod_pet
        FROM fact_production AS fp
        JOIN dim_time AS dt
            ON fp.time_key = dt.time_key
        JOIN (
            SELECT
                fp2.well_key,
                MIN(dt2.full_date) AS first_date
            FROM fact_production AS fp2
            JOIN dim_time AS dt2
                ON fp2.time_key = dt2.time_key
            GROUP BY fp2.well_key
        ) AS first_date
            ON fp.well_key = first_date.well_key
            AND dt.full_date = first_date.first_date
    ) AS first_prod
    JOIN (
        SELECT
            fp.well_key,
            dt.full_date AS last_date,
            fp.prod_pet AS last_prod_pet
        FROM fact_production AS fp
        JOIN dim_time AS dt
            ON fp.time_key = dt.time_key
        JOIN (
            SELECT
                fp2.well_key,
                MAX(dt2.full_date) AS last_date
            FROM fact_production AS fp2
            JOIN dim_time AS dt2
                ON fp2.time_key = dt2.time_key
            GROUP BY fp2.well_key
        ) AS last_date
            ON fp.well_key = last_date.well_key
            AND dt.full_date = last_date.last_date
    ) AS last_prod
        ON first_prod.well_key = last_prod.well_key
    WHERE NOT (
        first_prod.first_prod_pet = 0
        AND last_prod.last_prod_pet = 0
    )
) AS oil_decline
WHERE oil_decline_pct_per_month IS NOT NULL
ORDER BY oil_decline_pct_per_month DESC
LIMIT 10;
"""

In [12]:
df_bq03_oil = pd.read_sql(query_bq03_oil, conn)
df_bq03_oil

C:\Users\Admin\AppData\Local\Temp\ipykernel_13132\2295603044.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_bq03_oil = pd.read_sql(query_bq03_oil, conn)


,well_key,first_date,last_date,duration_months,first_prod_pet,last_prod_pet,oil_decline_absolute,oil_decline_pct,oil_decline_pct_per_month
0,9086,2026-04-30,2026-06-30,2,2212.46,0.00,2212.46,100.000000,50.000000
1,8882,2026-01-31,2026-06-30,4,4112.62,0.00,4112.62,100.000000,25.000000
2,10101,2026-02-28,2026-06-30,4,1666.00,320.73,1345.27,80.748499,20.187125
3,8417,2025-12-31,2026-06-30,5,18.71,0.00,18.71,100.000000,20.000000
4,8474,2026-04-30,2026-06-30,2,438.19,303.87,134.32,30.653370,15.326685
5,8389,2025-11-30,2026-06-30,7,369.76,18.64,351.12,94.958892,13.565556
6,8329,2025-08-31,2026-06-30,9,4577.84,40.60,4537.24,99.113119,11.012569
7,8326,2025-08-31,2026-06-30,9,287.41,6.11,281.30,97.874117,10.874902
8,8327,2025-08-31,2026-06-30,9,414.68,13.18,401.50,96.821646,10.757961
9,8348,2025-09-30,2026-06-30,9,256.91,21.47,235.44,91.642988,10.182554


In [13]:
query_bq03_gas = """
SELECT
    well_key,
    first_date,
    last_date,
    duration_months,
    first_prod_gas,
    last_prod_gas,
    gas_decline_absolute,
    gas_decline_pct,
    gas_decline_pct_per_month
FROM (
    SELECT
        first_prod.well_key,
        first_prod.first_date,
        first_prod.first_prod_gas,
        last_prod.last_date,
        last_prod.last_prod_gas,
        TIMESTAMPDIFF(
            MONTH,
            first_prod.first_date,
            last_prod.last_date
        ) AS duration_months,
        first_prod.first_prod_gas - last_prod.last_prod_gas
            AS gas_decline_absolute,
        CASE
            WHEN first_prod.first_prod_gas > 0
            THEN (
                (first_prod.first_prod_gas - last_prod.last_prod_gas)
                / first_prod.first_prod_gas
            ) * 100
            ELSE NULL
        END AS gas_decline_pct,
        CASE
            WHEN first_prod.first_prod_gas > 0
                AND TIMESTAMPDIFF(
                    MONTH,
                    first_prod.first_date,
                    last_prod.last_date
                ) > 0
            THEN (
                (
                    first_prod.first_prod_gas - last_prod.last_prod_gas
                )
                / first_prod.first_prod_gas
            ) * 100
            / TIMESTAMPDIFF(
                MONTH,
                first_prod.first_date,
                last_prod.last_date
            )
            ELSE NULL
        END AS gas_decline_pct_per_month
    FROM (
        SELECT
            fp.well_key,
            dt.full_date AS first_date,
            fp.prod_gas AS first_prod_gas
        FROM fact_production AS fp
        JOIN dim_time AS dt
            ON fp.time_key = dt.time_key
        JOIN (
            SELECT
                fp2.well_key,
                MIN(dt2.full_date) AS first_date
            FROM fact_production AS fp2
            JOIN dim_time AS dt2
                ON fp2.time_key = dt2.time_key
            GROUP BY fp2.well_key
        ) AS first_date
            ON fp.well_key = first_date.well_key
            AND dt.full_date = first_date.first_date
    ) AS first_prod
    JOIN (
        SELECT
            fp.well_key,
            dt.full_date AS last_date,
            fp.prod_gas AS last_prod_gas
        FROM fact_production AS fp
        JOIN dim_time AS dt
            ON fp.time_key = dt.time_key
        JOIN (
            SELECT
                fp2.well_key,
                MAX(dt2.full_date) AS last_date
            FROM fact_production AS fp2
            JOIN dim_time AS dt2
                ON fp2.time_key = dt2.time_key
            GROUP BY fp2.well_key
        ) AS last_date
            ON fp.well_key = last_date.well_key
            AND dt.full_date = last_date.last_date
    ) AS last_prod
        ON first_prod.well_key = last_prod.well_key
    WHERE NOT (
        first_prod.first_prod_gas = 0
        AND last_prod.last_prod_gas = 0
    )
) AS gas_decline
WHERE gas_decline_pct_per_month IS NOT NULL
ORDER BY gas_decline_pct_per_month DESC
LIMIT 10;
"""

In [14]:
df_bq03_gas = pd.read_sql(query_bq03_gas, conn)
df_bq03_gas

C:\Users\Admin\AppData\Local\Temp\ipykernel_13132\1751233509.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_bq03_gas = pd.read_sql(query_bq03_gas, conn)


,well_key,first_date,last_date,duration_months,first_prod_gas,last_prod_gas,gas_decline_absolute,gas_decline_pct,gas_decline_pct_per_month
0,9086,2026-04-30,2026-06-30,2,547.38,0.00,547.38,100.000000,50.000000
1,8882,2026-01-31,2026-06-30,4,806.83,0.00,806.83,100.000000,25.000000
2,10101,2026-02-28,2026-06-30,4,112.20,22.39,89.81,80.044563,20.011141
3,8417,2025-12-31,2026-06-30,5,2.31,0.00,2.31,100.000000,20.000000
4,8476,2026-04-30,2026-06-30,2,988.17,612.34,375.83,38.032930,19.016465
5,8389,2025-11-30,2026-06-30,7,30.97,2.32,28.65,92.508880,13.215554
6,8329,2025-08-31,2026-06-30,9,373.56,4.44,369.12,98.811436,10.979048
7,8326,2025-08-31,2026-06-30,9,37.47,1.13,36.34,96.984254,10.776028
8,8327,2025-08-31,2026-06-30,9,54.65,4.66,49.99,91.473010,10.163668
9,8348,2025-09-30,2026-06-30,9,31.77,3.12,28.65,90.179415,10.019935


### Interpretation

The results show that several wells exhibit rapid declines in both oil and gas production over their available production history.

For oil production, wells 9086 and 8882 show the highest observed decline rates, with 50.00% and 25.00% per month, respectively. Both wells reached zero oil production at the end of the available observation period. Well 10101 also shows a relatively high decline rate of 20.19% per month, decreasing from 1,666.00 to 320.73 units of oil production.

A similar pattern is observed for gas production. Wells 9086 and 8882 again show the highest decline rates, at 50.00% and 25.00% per month, respectively. Well 10101 shows a gas decline rate of 20.01% per month, while well 8476 shows a decline rate of 19.02% per month.

Several wells appear among the highest-decline wells for both oil and gas, suggesting that some wells experience a rapid reduction in overall hydrocarbon production rather than a decline limited to a single hydrocarbon.

However, most of the wells with the highest observed decline rates have relatively short production histories. For example, wells 9086, 8882, 10101, and 8417 have between two and five months of available production history. Therefore, these decline rates should be interpreted cautiously, as short observation periods can produce high monthly decline values that may not represent long-term production behavior.

Overall, the analysis identifies wells with rapid observed production declines, while also showing that production history duration is an important factor when interpreting decline rates.


### Business Conclusion

The analysis identifies wells with the highest observed production decline rates based on their available production history. Several wells exhibit rapid declines in both oil and gas production, with wells 9086 and 8882 showing the highest decline rates.

However, most of the wells with the highest decline rates have relatively short production histories. Therefore, these results should be considered an initial characterization of production decline rather than evidence of long-term well underperformance.

The results highlight the importance of considering production history duration when evaluating well decline and provide a basis for further analysis of production behavior during the early stages of well life.


## BQ-04 How does production evolve during the first months of a well's life?

### Business Question

How does oil and gas production evolve during the first months of a well's life?

### Analytical Approach

Analyze the production behavior of each well during the first months of its available production history.

For each well, production records will be ordered chronologically and the initial months of production will be compared to identify how oil and gas output changes during the early stage of the well's life.

Oil and gas production will be analyzed separately because they are measured in different units.

The analysis will focus on production by month since the beginning of each well's production history, allowing wells with different starting dates to be compared based on their relative production age rather than calendar time.

The results will be aggregated across wells to identify the general production pattern during the first months of well life.

In [15]:
bq04_production_first_12_months = """
SELECT
    production_months_relative.live_months,
    AVG(prod_pet) AS avg_pet_production,
    AVG(prod_gas) AS avg_gas_production,
    COUNT(DISTINCT production_months_relative.well_key) AS count_wells,
    COUNT(
        DISTINCT CASE
            WHEN wells_month_12.well_key IS NOT NULL
            THEN production_months_relative.well_key
        END
    ) AS wells_reaching_month_12
FROM (
    SELECT 
        full_date,
        fp.well_key,
        initial_date,
        prod_pet,
        prod_gas,
        TIMESTAMPDIFF(
            MONTH,
            initial_date,
            full_date
        ) AS live_months
        
    FROM (
        SELECT
            well_key,
            MIN(full_date) AS initial_date
        FROM fact_production AS fp
        JOIN dim_time AS dt
            ON fp.time_key = dt.time_key
        GROUP BY well_key
    ) AS initial_dates
    
    JOIN fact_production AS fp
        ON initial_dates.well_key = fp.well_key
    
    JOIN dim_time AS dt
        ON dt.time_key = fp.time_key

) AS production_months_relative

LEFT JOIN (
    SELECT DISTINCT
        fp.well_key
    FROM (
        SELECT
            well_key,
            MIN(full_date) AS initial_date
        FROM fact_production AS fp
        JOIN dim_time AS dt
            ON fp.time_key = dt.time_key
        GROUP BY well_key
    ) AS initial_dates
    
    JOIN fact_production AS fp
        ON initial_dates.well_key = fp.well_key
    
    JOIN dim_time AS dt
        ON dt.time_key = fp.time_key
    
    WHERE TIMESTAMPDIFF(
        MONTH,
        initial_date,
        full_date
    ) = 11

) AS wells_month_12

ON production_months_relative.well_key = wells_month_12.well_key

WHERE production_months_relative.live_months < 12

GROUP BY production_months_relative.live_months

ORDER BY production_months_relative.live_months;
"""

In [16]:
BQ_04 = pd.read_sql(bq04_production_first_12_months, conn)
BQ_04

C:\Users\Admin\AppData\Local\Temp\ipykernel_13132\4098392018.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  BQ_04 = pd.read_sql(bq04_production_first_12_months, conn)


,live_months,avg_pet_production,avg_gas_production,count_wells,wells_reaching_month_12
0,0,527.442826,793.682794,5041,2700
1,1,1355.098271,1652.086902,2655,1724
2,2,1789.891333,2005.367918,4078,2645
3,3,1821.796185,2036.861657,3055,1432
4,4,1610.486765,1877.386290,3270,2681
5,5,1510.789007,1935.056196,3731,1718
6,6,1238.617356,1780.148990,2829,2359
7,7,1149.191681,1632.176795,4303,2693
8,8,1064.603866,1682.899885,2550,1374
9,9,924.912647,1520.472722,3495,2700


#### Result

During the first 12 months of production, average oil and gas production fluctuates across relative well-life months. Average production reaches its highest levels around months 2 and 3 and then shows an overall declining trend toward month 11.

A total of 5,041 wells have production records during their first month (`live_months = 0`), while 2,700 wells have production records during the twelfth month (`live_months = 11`).

Among the 5,041 wells observed during their first month, 2,700 also have production records during the twelfth month, representing approximately 53.6% of the initial group.

### Interpretation

The results suggest that wells experience an initial increase in average production during the first few months, followed by an overall decline in average oil and gas production.

However, the number of wells with available production records varies considerably across relative well-life months. Therefore, the observed changes in average production should not be interpreted as the evolution of exactly the same population of wells throughout the entire 12-month period.

In addition, the absence of a production record during the twelfth month does not necessarily indicate that a well stopped producing. It may reflect other characteristics or limitations of the dataset. Therefore, the percentage of wells observed in month 11 should be interpreted as **observation coverage**, rather than as a well abandonment or production discontinuation rate.


### Business Conclusion

The analysis of the first 12 months of production shows that average oil and gas production tends to increase during the initial months and subsequently follows an overall declining trend. This pattern provides an initial characterization of well production behavior during the first year of available production history.

However, the number of observed wells varies across relative production months, and only 53.6% of the wells observed during their first month also have records during the twelfth month. Consequently, the evolution of average production should be interpreted while considering changes in the observed well population.

This analysis provides a baseline characterization of early well production performance, but it does not by itself establish whether geological or operational factors explain differences in production. These factors will be investigated in the following Business Questions.

## BQ-05 Do wells follow similar production decline patterns over time?

### Business Question

Do wells follow similar production decline patterns over time?

### Analytical Approach

Analyze the production trajectories of individual wells over their relative production age to determine whether wells exhibit similar decline patterns over time.

Production will be analyzed separately for oil and gas because they are measured in different units.

For each well, production records will be ordered according to the number of months elapsed since the beginning of its available production history. The analysis will focus on the first 12 relative production months to provide a common observation window across wells with different production start dates.

The production trajectory of each well will then be compared with the trajectories of other wells to identify whether wells generally follow similar decline patterns or exhibit substantial differences in their production behavior.

### SQL

In [ ]:
query_bq05_stage1_absolute_trajectory = """
WITH production_monthly AS (

    SELECT
        well_key,
        live_months,
        SUM(prod_pet) AS pet_prod_total,
        SUM(prod_gas) AS gas_prod_total
    FROM (
        SELECT
            fp.well_key,
            TIMESTAMPDIFF(
                MONTH,
                initial_dates.first_date,
                dt.full_date
            ) AS live_months,
            fp.prod_pet,
            fp.prod_gas
        FROM (
            SELECT
                fp.well_key,
                MIN(dt.full_date) AS first_date
            FROM fact_production AS fp
            JOIN dim_time AS dt
                ON fp.time_key = dt.time_key
            GROUP BY fp.well_key
        ) AS initial_dates
        JOIN fact_production AS fp
            ON initial_dates.well_key = fp.well_key
        JOIN dim_time AS dt
            ON fp.time_key = dt.time_key
    ) AS prod_well
    WHERE live_months < 12
    GROUP BY
        well_key,
        live_months
),

oil_ranked AS (
    SELECT
        live_months,
        pet_prod_total,
        ROW_NUMBER() OVER (
            PARTITION BY live_months
            ORDER BY pet_prod_total
        ) AS rn,
        COUNT(*) OVER (
            PARTITION BY live_months
        ) AS n
    FROM production_monthly
    WHERE pet_prod_total IS NOT NULL
),

gas_ranked AS (
    SELECT
        live_months,
        gas_prod_total,
        ROW_NUMBER() OVER (
            PARTITION BY live_months
            ORDER BY gas_prod_total
        ) AS rn,
        COUNT(*) OVER (
            PARTITION BY live_months
        ) AS n
    FROM production_monthly
    WHERE gas_prod_total IS NOT NULL
),

oil_summary AS (
    SELECT
        live_months,
        COUNT(*) AS oil_wells,
        AVG(
            CASE
                WHEN rn IN (
                    FLOOR((n + 1) * 0.25),
                    CEIL((n + 1) * 0.25)
                )
                THEN pet_prod_total
            END
        ) AS oil_p25,
        AVG(
            CASE
                WHEN rn IN (
                    FLOOR((n + 1) * 0.50),
                    CEIL((n + 1) * 0.50)
                )
                THEN pet_prod_total
            END
        ) AS oil_median,
        AVG(
            CASE
                WHEN rn IN (
                    FLOOR((n + 1) * 0.75),
                    CEIL((n + 1) * 0.75)
                )
                THEN pet_prod_total
            END
        ) AS oil_p75
    FROM oil_ranked
    GROUP BY live_months
),

gas_summary AS (
    SELECT
        live_months,
        COUNT(*) AS gas_wells,
        AVG(
            CASE
                WHEN rn IN (
                    FLOOR((n + 1) * 0.25),
                    CEIL((n + 1) * 0.25)
                )
                THEN gas_prod_total
            END
        ) AS gas_p25,
        AVG(
            CASE
                WHEN rn IN (
                    FLOOR((n + 1) * 0.50),
                    CEIL((n + 1) * 0.50)
                )
                THEN gas_prod_total
            END
        ) AS gas_median,
        AVG(
            CASE
                WHEN rn IN (
                    FLOOR((n + 1) * 0.75),
                    CEIL((n + 1) * 0.75)
                )
                THEN gas_prod_total
            END
        ) AS gas_p75
    FROM gas_ranked
    GROUP BY live_months
)

SELECT
    o.live_months,
    o.oil_wells,
    o.oil_p25,
    o.oil_median,
    o.oil_p75,
    g.gas_wells,
    g.gas_p25,
    g.gas_median,
    g.gas_p75
FROM oil_summary AS o
LEFT JOIN gas_summary AS g
    ON o.live_months = g.live_months
ORDER BY o.live_months;
"""

In [18]:
BQ_05_result_I = pd.read_sql(query_bq05_stage1_absolute_trajectory, conn)
BQ_05_result_I

C:\Users\Admin\AppData\Local\Temp\ipykernel_13132\3450875938.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  BQ_05_result_I = pd.read_sql(query_bq05_stage1_absolute_trajectory, conn)


,live_months,oil_wells,oil_p25,oil_median,oil_p75,gas_wells,gas_p25,gas_median,gas_p75
0,0,5041,0.00,18.800,619.950,5041,0.000,113.900,735.805
1,1,2655,0.00,395.260,2415.020,2655,122.430,528.520,2033.900
2,2,4078,0.00,385.300,4037.055,4078,193.835,777.805,2498.105
3,3,3055,0.00,522.710,4563.350,3055,307.350,945.850,2763.820
4,4,3270,0.00,343.845,3631.695,3270,206.575,727.555,2301.300
5,5,3731,1.02,473.350,3773.870,3731,266.640,866.450,2590.570
6,6,2829,0.00,196.570,2420.260,2829,145.435,558.130,1842.225
7,7,4303,0.00,308.720,2627.940,4303,175.410,634.810,2275.590
8,8,2550,2.97,300.895,2392.200,2550,174.365,622.320,2069.565
9,9,3495,0.00,193.040,1987.610,3495,128.570,478.710,1869.370


In [ ]:
query_bq05_stage2_peak_month_distribution = """
WITH production_monthly AS (
    SELECT
        well_key,
        live_months,
        SUM(prod_pet) AS pet_prod_total,
        SUM(prod_gas) AS gas_prod_total
    FROM (
        SELECT
            fp.well_key,
            TIMESTAMPDIFF(
                MONTH,
                initial_dates.first_date,
                dt.full_date
            ) AS live_months,
            fp.prod_pet,
            fp.prod_gas
        FROM (
            SELECT
                fp.well_key,
                MIN(dt.full_date) AS first_date
            FROM fact_production AS fp
            JOIN dim_time AS dt
                ON fp.time_key = dt.time_key
            GROUP BY fp.well_key
        ) AS initial_dates
        JOIN fact_production AS fp
            ON initial_dates.well_key = fp.well_key
        JOIN dim_time AS dt
            ON fp.time_key = dt.time_key
    ) AS prod_well
    WHERE live_months < 12
    GROUP BY
        well_key,
        live_months
),

oil_peak AS (
    SELECT
        well_key,
        live_months AS oil_peak_month,
        pet_prod_total AS oil_peak_production,
        ROW_NUMBER() OVER (
            PARTITION BY well_key
            ORDER BY
                pet_prod_total DESC,
                live_months ASC
        ) AS rn
    FROM production_monthly
    WHERE pet_prod_total > 0
),

gas_peak AS (
    SELECT
        well_key,
        live_months AS gas_peak_month,
        gas_prod_total AS gas_peak_production,
        ROW_NUMBER() OVER (
            PARTITION BY well_key
            ORDER BY
                gas_prod_total DESC,
                live_months ASC
        ) AS rn
    FROM production_monthly
    WHERE gas_prod_total > 0
),

peak_by_well AS (
    SELECT
        o.well_key,
        o.oil_peak_month,
        o.oil_peak_production,
        g.gas_peak_month,
        g.gas_peak_production
    FROM oil_peak AS o
    LEFT JOIN gas_peak AS g
        ON o.well_key = g.well_key
    WHERE o.rn = 1
        AND g.rn = 1
),

oil_distribution AS (
    SELECT
        oil_peak_month AS live_months,
        COUNT(*) AS oil_peak_wells,
        ROUND(
            COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
            2
        ) AS oil_peak_percentage
    FROM peak_by_well
    GROUP BY oil_peak_month
),

gas_distribution AS (
    SELECT
        gas_peak_month AS live_months,
        COUNT(*) AS gas_peak_wells,
        ROUND(
            COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
            2
        ) AS gas_peak_percentage
    FROM peak_by_well
    GROUP BY gas_peak_month
)

SELECT
    COALESCE(o.live_months, g.live_months) AS live_months,
    COALESCE(o.oil_peak_wells, 0) AS oil_peak_wells,
    COALESCE(o.oil_peak_percentage, 0) AS oil_peak_percentage,
    COALESCE(g.gas_peak_wells, 0) AS gas_peak_wells,
    COALESCE(g.gas_peak_percentage, 0) AS gas_peak_percentage
FROM oil_distribution AS o
LEFT JOIN gas_distribution AS g
    ON o.live_months = g.live_months
ORDER BY live_months;
"""

In [20]:
BQ_05_result_II = pd.read_sql(query_bq05_stage2_peak_month_distribution, conn)
BQ_05_result_II

C:\Users\Admin\AppData\Local\Temp\ipykernel_13132\3841231172.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  BQ_05_result_II = pd.read_sql(query_bq05_stage2_peak_month_distribution, conn)


,live_months,oil_peak_wells,oil_peak_percentage,gas_peak_wells,gas_peak_percentage
0,0,356,8.84,290,7.20
1,1,463,11.49,409,10.15
2,2,978,24.27,962,23.88
3,3,694,17.23,740,18.37
4,4,406,10.08,439,10.90
5,5,276,6.85,313,7.77
6,6,73,1.81,67,1.66
7,7,306,7.59,357,8.86
8,8,92,2.28,95,2.36
9,9,197,4.89,185,4.59


### Interpretation

The absolute production analysis shows a broad production trajectory characterized by an early increase in production followed by a decline during the first 12 relative production months. However, the first relative month should be interpreted with caution because it represents the period containing each well's first recorded production date and may not correspond to a complete production month.

The max-normalized analysis provides a clearer view of the common production trajectory across wells. Both oil and gas production generally increase during the first few relative months, reach their highest levels around months 2–3, and subsequently decline.

The distribution of peak production months provides additional support for this pattern. The largest proportion of wells reaches maximum production around relative month 2, followed by month 3. Nevertheless, peak production is distributed across the entire 12-month window, indicating substantial heterogeneity in individual well production trajectories.

The analysis should not be interpreted as evidence that all wells follow the same production profile. Rather, it indicates the presence of a broad common pattern combined with significant well-level variability. Missing relative months represent lack of observed production records rather than zero production.

### Business Conclusion

Wells exhibit a broadly similar production trajectory during their first 12 relative production months. Production generally increases during the early months, reaches a peak around the second and third relative months, and then declines.

The concentration of peak production around months 2–3 supports the presence of a common early-life production pattern across a substantial proportion of wells. However, the distribution of peak months across the full 12-month window and the spread observed in the normalized production trajectories demonstrate considerable heterogeneity between individual wells.

Therefore, while wells share a general early production and decline pattern, their individual production trajectories are not uniform. This variability suggests that subsequent analyses should investigate the operational, geological, and reservoir characteristics that may explain differences in well performance.

## BQ-Pregunta
### Business Question
### Analytical Approach
### SQL
### Interpretation
### Business Conclusion
